# Experimentos — Gossip Simulator

**Objetivo:** comparar Push, Push-Pull, Push + anti-entropy e fanout adaptativo + anti-entropy em **cobertura, T50/T95 e tráfego**, usando o mesmo modelo de rede e o mesmo orçamento de bytes.

**Hipótese:** a política adaptativa pode reduzir tráfego redundante sem degradar significativamente cobertura e T95.

> **Template de experimento:** as células de execução dependem da função `run_many` que será implementada em `experiments/helper.py` e do motor em `src/`. As células de análise e visualização usam o contrato de DataFrames documentado no README.

## 1. Imports e configuração

Usar parâmetros declarativos no YAML. O notebook **não implementa** regras de Gossip, transporte, coleta de métricas ou agendamento de eventos.

In [ ]:
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

sns.set_theme(style="whitegrid")

# Jupyter normalmente inicia o kernel em experiments/; aceita também iniciar na raiz.
ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent

with (ROOT / "configs" / "baseline.yaml").open(encoding="utf-8") as f:
    BASE_CONFIG = yaml.safe_load(f)

SEEDS = list(range(30))
BASE_CONFIG

## 2. Tratamentos (protocolos)

**Uma diferença de cada vez:** manter topologia, falhas, latências e orçamento idênticos entre tratamentos. As configurações abaixo são o contrato **proposto** do `helper.py`.

In [ ]:
VARIANTS = {
    "push_f1": {
        "protocol": "push", "fanout": 1, "anti_entropy": False,
    },
    "push_f3": {
        "protocol": "push", "fanout": 3, "anti_entropy": False,
    },
    "push_ae": {
        "protocol": "push", "fanout": 1, "anti_entropy": True,
    },
    "adaptive_ae": {
        "protocol": "push", "fanout": "adaptive", "anti_entropy": True,
    },
}

VARIANTS

## 3. Cenários

Começar por uma rede sem falhas para validar o Push contra a tendência teórica. Em seguida, introduzir perda, heterogeneidade, churn e partição separadamente.

In [ ]:
SCENARIOS = {
    "stable": {},
    "loss_5pct": {"packet_loss": 0.05},
    "heterogeneous_latency": {"latency_ms": [10, 200]},
    "churn_10pct": {"churn_fraction": 0.10},
    "temporary_partition": {
        "partition": {"start_ms": 10000, "end_ms": 30000, "groups": 2},
    },
}


def make_config(network_overrides):
    config = deepcopy(BASE_CONFIG)
    config["network"].update(network_overrides)
    return config

make_config(SCENARIOS["loss_5pct"])

## 4. Smoke test: uma execução pequena

Antes das 30 repetições, rodar com **50 nós e 1 seed**. Verificar se cobertura fica em `[0, 1]`, se os tempos não são negativos e se o orçamento está sendo respeitado.

> **A implementar:** `experiments/helper.py::run_many(config, variants, seeds, scenario) -> (summary_df, coverage_df)`.
>
> `summary_df`: `scenario, protocol, seed, n_nodes, coverage_final, t50_ms, t95_ms, bytes_total, bytes_control, duplicates`.
>
> `coverage_df`: `scenario, protocol, seed, time_ms, coverage`.

In [ ]:
# DESCOMENTE após implementar `run_many` em experiments/helper.py.
# from helper import run_many
#
# smoke_config = make_config(SCENARIOS["stable"])
# smoke_config["network"]["nodes"] = 50
# smoke_summary, smoke_curve = run_many(
#     config=smoke_config,
#     variants={"push_f1": VARIANTS["push_f1"]},
#     seeds=[0],
#     scenario="smoke",
# )
# display(smoke_summary)
# display(smoke_curve.head())

## 5. Executar o grid experimental

Usar **as mesmas seeds** por cenário em todos os protocolos. No motor, separar as fontes aleatórias por topologia, falhas, latência e escolhas dos nós: assim, mudar o fanout não altera acidentalmente os eventos externos de uma execução.

In [ ]:
# DESCOMENTE após validar o smoke test.
# from helper import run_many
#
# summaries = []
# curves = []
# for scenario, overrides in SCENARIOS.items():
#     summary, curve = run_many(
#         config=make_config(overrides),
#         variants=VARIANTS,
#         seeds=SEEDS,
#         scenario=scenario,
#     )
#     summaries.append(summary)
#     curves.append(curve)
#
# summary_df = pd.concat(summaries, ignore_index=True)
# coverage_df = pd.concat(curves, ignore_index=True)

## 6. Verificações e qualidade de dados

Não descartar execuções que falham em atingir 95% de cobertura: registrar `t95_ms = NaN` e reportar também a **fração que atingiu 95%**. Se houver churn, definir a coorte elegível. Confirme se todos os protocolos obedecem ao orçamento (dados + controle + reparos).

In [ ]:
# Executar depois de criar summary_df e coverage_df:
# assert summary_df["coverage_final"].between(0, 1).all()
# assert coverage_df["coverage"].between(0, 1).all()
#
# print("Executados:", len(summary_df))
# display(summary_df.isna().sum())
# reached = (
#     summary_df.assign(reached_t95=summary_df["t95_ms"].notna())
#     .groupby(["scenario", "protocol"])["reached_t95"]
#     .mean()
#     .rename("fraction_reaching_t95")
# )
# display(reached.reset_index())

## 7. Figuras

Primeiro comparar cobertura × tempo; depois T95 e bytes. Evitar comparar apenas T95 quando há tratamentos que nunca atingiram 95%.

In [ ]:
# Executar depois de criar coverage_df:
# data = coverage_df.query("scenario == 'stable'")
# fig, ax = plt.subplots(figsize=(8, 4))
# sns.lineplot(
#     data=data,
#     x="time_ms", y="coverage", hue="protocol",
#     estimator="mean", errorbar=("ci", 95), ax=ax,
# )
# ax.set(xlabel="Tempo virtual (ms)", ylabel="Cobertura (fração)", ylim=(0, 1.02))
# plt.show()

In [ ]:
# Executar depois de criar summary_df:
# data = summary_df.query("scenario == 'loss_5pct'")
# fig, ax = plt.subplots(figsize=(8, 4))
# sns.scatterplot(
#     data=data, x="bytes_total", y="t95_ms",
#     hue="protocol", alpha=0.6, ax=ax,
# )
# ax.set(xlabel="Bytes totais enviados", ylabel="T95 (ms)")
# plt.show()
#
# # Complementar com tabelas de cobertura final e fração de runs que atingiram T95.

## 8. Tabelas, dispersão e análise

Agregar por cenário e protocolo. Reportar mediana, dispersão (por exemplo, quantis), fração de execuções com T95 e casos em que o adaptativo **perde** para algum baseline. Se necessário, usar SciPy para intervalos de confiança e testes apropriados. Separar seeds de ajuste (Optuna) das de avaliação final.

In [ ]:
# Executar depois de criar summary_df:
# comparison = (
#     summary_df
#     .groupby(["scenario", "protocol"], as_index=False)
#     .agg(
#         coverage_median=("coverage_final", "median"),
#         bytes_median=("bytes_total", "median"),
#         t95_median_reached=("t95_ms", "median"),
#         t95_reached_rate=("t95_ms", lambda s: s.notna().mean()),
#     )
# )
# display(comparison)

## 9. Exportar resultados e registrar conclusões

Salvar dados brutos, configurações e metadados (seeds e versão do código) para reproduzir gráficos sem rodar tudo novamente. A análise final deve responder: **o adaptativo reduz tráfego sem piorar T95 ou cobertura? Em quais cenários falha?**

In [ ]:
# Executar depois de criar os DataFrames:
# output_dir = ROOT / "results"
# output_dir.mkdir(exist_ok=True)
# summary_df.to_csv(output_dir / "summary.csv", index=False)
# coverage_df.to_csv(output_dir / "coverage.csv", index=False)
#
# with (output_dir / "experiment_config.yaml").open("w", encoding="utf-8") as f:
#     yaml.safe_dump({"base": BASE_CONFIG, "variants": VARIANTS,
#                     "scenarios": SCENARIOS, "seeds": SEEDS}, f, sort_keys=False)

### Conclusões (preencher após rodar)

- **Validação do baseline:** ...
- **Cobertura e T95:** ...
- **Tráfego / duplicação:** ...
- **Casos sem ganho do adaptativo:** ...
- **Limitações:** modelo de perdas, topologias, sem rede real e sensibilidade aos parâmetros.
